In [ ]:
CODE_DATASET  = '/kaggle/input/datasets/justagek/dloc-code'
TRAIN_DATASET = '/kaggle/input/datasets/justagek/dloc-data-train'
TEST_DATASET  = '/kaggle/input/datasets/justagek/dloc-data-test'
CKPT_DATASET    = '/kaggle/input/datasets/justagek/last-epoch-27'  

WORK_DIR      = '/kaggle/working/dloc'
RUNS_DIR      = '/kaggle/working/runs'

N_EPOCHS      = 50
BATCH_SIZE    = 32
GPU_IDS       = ['0', '1']

In [ ]:
import os, shutil, glob

os.makedirs(f'{WORK_DIR}/data', exist_ok=True)

# --- 1. Copy Python source files ---
for f in os.listdir(CODE_DATASET):
    if f.endswith('.py'):
        shutil.copy(os.path.join(CODE_DATASET, f), WORK_DIR)
print('Code copied.')

# --- 2. Symlink data files ---
for dataset_dir in [TRAIN_DATASET, TEST_DATASET]:
    for f in os.listdir(dataset_dir):
        if f.endswith('.mat'):
            dst = f'{WORK_DIR}/data/{f}'
            if not os.path.exists(dst):
                os.symlink(os.path.join(dataset_dir, f), dst)
print('Data linked.')
print('Files in data/:', os.listdir(f'{WORK_DIR}/data/'))

# --- 3. Find checkpoint from dataset (recursive search) ---
RESUME_EPOCH = 0
ckpt_dst = f'{WORK_DIR}/checkpoint'
components = ['encoder', 'decoder', 'offset_decoder']

if os.path.exists(CKPT_DATASET):
    # Search recursively for the highest numbered epoch
    all_pths = glob.glob(f'{CKPT_DATASET}/**/encoder/*.pth', recursive=True)
    best_epoch = 0
    best_dir = None
    for p in all_pths:
        fname = os.path.basename(p)
        parent_dir = os.path.dirname(os.path.dirname(p))
        try:
            epoch_num = int(fname.split('_')[0])
            if epoch_num > best_epoch:
                best_epoch = epoch_num
                best_dir = parent_dir
        except ValueError:
            pass

    if best_epoch > 0:
        RESUME_EPOCH = best_epoch
        print(f'Found epoch {best_epoch} checkpoint at: {best_dir}')

        # Copy checkpoint files to working directory
        for comp in components:
            os.makedirs(f'{ckpt_dst}/{comp}', exist_ok=True)
            comp_dir = f'{best_dir}/{comp}'
            if os.path.isdir(comp_dir):
                for pth in glob.glob(f'{comp_dir}/*.pth'):
                    shutil.copy(pth, f'{ckpt_dst}/{comp}/')
                print(f'  {comp}: {os.listdir(f"{ckpt_dst}/{comp}/")}')
            else:
                print(f'  WARNING: {comp_dir} not found!')
    else:
        print('Checkpoint dataset exists but no numbered checkpoints found.')
else:
    print(f'No checkpoint dataset found at {CKPT_DATASET}')
    print('Will train from scratch (epoch 0).')

print(f'\nWill resume from epoch {RESUME_EPOCH}, train to epoch {N_EPOCHS}.')
print(f'Epochs remaining: {N_EPOCHS - RESUME_EPOCH}')

In [ ]:
import os
os.chdir(WORK_DIR)

# --- Fix 1: trainer.py — UnboundLocalError when resuming ---
with open('trainer.py', 'r') as f:
    code = f.read()

code = code.replace(
    'if (epoch==1):',
    'if (epoch == model.opt.starting_epoch_count + 1):'
)

with open('trainer.py', 'w') as f:
    f.write(code)
print('Fixed trainer.py (resume bug)')

# --- Fix 2: modelADT.py — weights_only for new PyTorch ---
with open('modelADT.py', 'r') as f:
    code = f.read()

if 'weights_only' not in code:
    code = code.replace(
        'state_dict = torch.load(load_path)',
        'state_dict = torch.load(load_path, weights_only=False)'
    )
    with open('modelADT.py', 'w') as f:
        f.write(code)
    print('Fixed modelADT.py (weights_only)')
else:
    print('modelADT.py already patched.')

# --- Fix 3: train_and_test.py — ensure all fixes are present ---
with open('train_and_test.py', 'r') as f:
    code = f.read()

# Fix: return -1 → sys.exit(-1)
if 'return -1' in code:
    code = code.replace('return -1', 'sys.exit(-1)')

# Fix: eval_name undefined
if 'eval_name = opt_exp.checkpoints_dir' not in code:
    code = code.replace(
        'epoch = "best"  # int/"best"/"last"',
        'epoch = "best"  # int/"best"/"last"\neval_name = opt_exp.checkpoints_dir'
    )

# Fix: isFrozen KeyError
if 'opt_exp.isFrozen' in code and 'getattr' not in code:
    code = code.replace(
        'if opt_exp.isFrozen:',
        'if getattr(opt_exp, \'isFrozen\', False):'
    )

with open('train_and_test.py', 'w') as f:
    f.write(code)
print('Fixed train_and_test.py')

In [ ]:
# Write params.py dynamically based on detected checkpoint
CONTINUE = RESUME_EPOCH > 0

params_code = f'''
from easydict import EasyDict as edict
import time
from os.path import join

opt_exp = edict()
opt_exp.isTrain = True
opt_exp.continue_train = {CONTINUE}
opt_exp.starting_epoch_count = {RESUME_EPOCH}
opt_exp.n_epochs = {N_EPOCHS}
opt_exp.gpu_ids = {GPU_IDS}
opt_exp.data = "rw_to_rw"
opt_exp.n_decoders = 2
opt_exp.batch_size = {BATCH_SIZE}
opt_exp.ds_step_trn = 1
opt_exp.ds_step_tst = 1
opt_exp.weight_decay = 1e-5
opt_exp.confidence = False
opt_exp.save_name = time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())
opt_exp.checkpoints_dir = join(\'{RUNS_DIR}\', opt_exp.save_name)
opt_exp.results_dir = opt_exp.checkpoints_dir
opt_exp.log_dir = opt_exp.checkpoints_dir
opt_exp.load_dir = \'{ckpt_dst if CONTINUE else ""}\'\n
opt_encoder = edict()
opt_encoder.parent_exp = opt_exp
opt_encoder.batch_size = opt_exp.batch_size
opt_encoder.ngf = 64
opt_encoder.base_model = \'resnet_encoder\'
opt_encoder.net = \'G\'
opt_encoder.resnet_blocks = 6
opt_encoder.no_dropout = False
opt_encoder.init_type = \'xavier\'
opt_encoder.init_gain = 0.02
opt_encoder.norm = \'instance\'
opt_encoder.beta1 = 0.5
opt_encoder.lr = 0.00001
opt_encoder.lr_policy = \'step\'
opt_encoder.lr_decay_iters = 50
opt_encoder.lambda_L = 1
opt_encoder.lambda_cross = 1e-5
opt_encoder.lambda_reg = 5e-4
opt_encoder.weight_decay = opt_exp.weight_decay
opt_encoder.input_nc = 4
opt_encoder.output_nc = 1
opt_encoder.save_latest_freq = 5000
opt_encoder.save_epoch_freq = 1
opt_encoder.n_epochs = opt_exp.n_epochs
opt_encoder.isTrain = True
opt_encoder.continue_train = {CONTINUE}
opt_encoder.starting_epoch_count = opt_exp.starting_epoch_count
opt_encoder.name = \'encoder\'
opt_encoder.loss_type = "NoLoss"
opt_encoder.niter = 25
opt_encoder.niter_decay = 25
opt_encoder.gpu_ids = opt_exp.gpu_ids
opt_encoder.num_threads = 4
opt_encoder.checkpoints_load_dir = opt_exp.load_dir
opt_encoder.checkpoints_save_dir = opt_exp.checkpoints_dir
opt_encoder.results_dir = opt_exp.results_dir
opt_encoder.log_dir = opt_exp.log_dir
opt_encoder.max_dataset_size = float("inf")
opt_encoder.verbose = False
opt_encoder.suffix = \'\'

opt_decoder = edict()
opt_decoder.parent_exp = opt_exp
opt_decoder.batch_size = opt_exp.batch_size
opt_decoder.ngf = 64
opt_decoder.base_model = \'resnet_decoder\'
opt_decoder.net = \'G\'
opt_decoder.resnet_blocks = 9
opt_decoder.encoder_res_blocks = 6
opt_decoder.no_dropout = False
opt_decoder.init_type = \'xavier\'
opt_decoder.init_gain = 0.02
opt_decoder.norm = \'instance\'
opt_decoder.beta1 = 0.5
opt_decoder.lr = opt_encoder.lr
opt_decoder.lr_policy = \'step\'
opt_decoder.lr_decay_iters = 20
opt_decoder.lambda_L = 1
opt_decoder.lambda_cross = 1e-5
opt_decoder.lambda_reg = 5e-4
opt_decoder.weight_decay = opt_exp.weight_decay
opt_decoder.input_nc = 4
opt_decoder.output_nc = 1
opt_decoder.save_latest_freq = 5000
opt_decoder.save_epoch_freq = 1
opt_decoder.n_epochs = opt_exp.n_epochs
opt_decoder.isTrain = True
opt_decoder.continue_train = {CONTINUE}
opt_decoder.starting_epoch_count = opt_exp.starting_epoch_count
opt_decoder.name = \'decoder\'
opt_decoder.loss_type = "L2_sumL1"
opt_decoder.niter = 25
opt_decoder.niter_decay = 25
opt_decoder.gpu_ids = opt_exp.gpu_ids
opt_decoder.num_threads = 4
opt_decoder.checkpoints_load_dir = opt_exp.load_dir
opt_decoder.checkpoints_save_dir = opt_exp.checkpoints_dir
opt_decoder.results_dir = opt_exp.results_dir
opt_decoder.log_dir = opt_exp.log_dir
opt_decoder.verbose = False
opt_decoder.suffix = \'\'

opt_offset_decoder = edict()
opt_offset_decoder.parent_exp = opt_exp
opt_offset_decoder.batch_size = opt_exp.batch_size
opt_offset_decoder.ngf = 64
opt_offset_decoder.base_model = \'resnet_decoder\'
opt_offset_decoder.net = \'G\'
opt_offset_decoder.resnet_blocks = 12
opt_offset_decoder.encoder_res_blocks = 6
opt_offset_decoder.no_dropout = False
opt_offset_decoder.init_type = \'xavier\'
opt_offset_decoder.init_gain = 0.02
opt_offset_decoder.norm = \'instance\'
opt_offset_decoder.beta1 = 0.5
opt_offset_decoder.lr = opt_encoder.lr
opt_offset_decoder.lr_policy = \'step\'
opt_offset_decoder.lr_decay_iters = 50
opt_offset_decoder.lambda_L = 1
opt_offset_decoder.lambda_cross = 0
opt_offset_decoder.lambda_reg = 0
opt_offset_decoder.weight_decay = opt_exp.weight_decay
opt_offset_decoder.input_nc = 4
opt_offset_decoder.output_nc = 4
opt_offset_decoder.save_latest_freq = 5000
opt_offset_decoder.save_epoch_freq = 1
opt_offset_decoder.n_epochs = opt_exp.n_epochs
opt_offset_decoder.isTrain = True
opt_offset_decoder.continue_train = {CONTINUE}
opt_offset_decoder.starting_epoch_count = opt_exp.starting_epoch_count
opt_offset_decoder.name = \'offset_decoder\'
opt_offset_decoder.loss_type = "L2_offset_loss"
opt_offset_decoder.niter = 25
opt_offset_decoder.niter_decay = 25
opt_offset_decoder.gpu_ids = opt_exp.gpu_ids
opt_offset_decoder.num_threads = 4
opt_offset_decoder.checkpoints_load_dir = opt_exp.load_dir
opt_offset_decoder.checkpoints_save_dir = opt_exp.checkpoints_dir
opt_offset_decoder.results_dir = opt_exp.results_dir
opt_offset_decoder.log_dir = opt_exp.log_dir
opt_offset_decoder.verbose = False
opt_offset_decoder.suffix = \'\'
'''.strip()

with open(f'{WORK_DIR}/params.py', 'w') as f:
    f.write(params_code)

print(f'params.py written:')
print(f'  starting_epoch_count = {RESUME_EPOCH}')
print(f'  n_epochs = {N_EPOCHS}')
print(f'  continue_train = {CONTINUE}')
print(f'  batch_size = {BATCH_SIZE}')
print(f'  gpu_ids = {GPU_IDS}')

In [ ]:
!pip install easydict hdf5storage h5py -q
!python train_and_test.py

In [ ]:
import glob, os

run_dirs = sorted(glob.glob(f'{RUNS_DIR}/*'))
if not run_dirs:
    print('No runs found!')
else:
    latest = run_dirs[-1]
    print(f'Run: {latest}\n')
    
    # Print all log files
    print('=== Logs ===')
    for txt in sorted(glob.glob(f'{latest}/*.txt')):
        name = os.path.basename(txt)
        with open(txt) as f:
            lines = f.readlines()
        last_val = lines[-1].strip() if lines else 'empty'
        print(f'  {name}: {len(lines)} entries, last = {last_val}')
    
    # Print key metrics
    print('\n=== Key Metrics ===')
    for metric in ['decoder_test_median_error', 'decoder_test_90_error', 'decoder_test_99_error']:
        path = f'{latest}/{metric}.txt'
        if os.path.exists(path):
            with open(path) as f:
                vals = [float(l.strip()) for l in f.readlines() if l.strip()]
            if vals:
                print(f'  {metric}:')
                for i, v in enumerate(vals):
                    epoch_num = RESUME_EPOCH + 1 + i
                    print(f'    epoch {epoch_num}: {v:.4f} m')
    
    # Print saved model files
    print('\n=== Saved Models ===')
    for comp in ['encoder', 'decoder', 'offset_decoder']:
        comp_dir = f'{latest}/{comp}'
        if os.path.isdir(comp_dir):
            files = sorted(os.listdir(comp_dir))
            sizes = [f'{os.path.getsize(os.path.join(comp_dir, f))/1e6:.1f}MB' for f in files]
            print(f'  {comp}: {len(files)} files')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

run_dirs = sorted(glob.glob(f'{RUNS_DIR}/*'))
if run_dirs:
    latest = run_dirs[-1]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    metrics = [
        ('decoder_test_median_error.txt', 'Test Median Error (m)', 0.64),
        ('decoder_test_90_error.txt', 'Test 90th Percentile (m)', 1.60),
        ('decoder_test_99_error.txt', 'Test 99th Percentile (m)', 3.20),
    ]
    
    for ax, (fname, title, paper_target) in zip(axes, metrics):
        path = f'{latest}/{fname}'
        if os.path.exists(path):
            with open(path) as f:
                vals = [float(l.strip()) for l in f.readlines() if l.strip()]
            epochs = list(range(RESUME_EPOCH + 1, RESUME_EPOCH + 1 + len(vals)))
            ax.plot(epochs, vals, 'b-o', markersize=3, label='Our run')
            ax.axhline(y=paper_target, color='r', linestyle='--', label=f'Paper target ({paper_target}m)')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Error (m)')
            ax.set_title(title)
            ax.legend()
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{latest}/training_progress.png', dpi=150)
    plt.show()
    print(f'Plot saved to {latest}/training_progress.png')